In [ ]:
import pandas as pd
import glob
import os

folder_path = "../data/raw"

files = glob.glob(os.path.join(folder_path, "*.csv"))

print("Znalezione pliki:")
print(files)
print("Liczba plików:", len(files))

if len(files) == 0:
    raise FileNotFoundError("Nie znaleziono plików CSV. Sprawdź, czy dane są w folderze data/raw.")

df_list = []

for file in files:
    temp = pd.read_csv(file)
    temp["source_file"] = os.path.basename(file)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df["year_month"] = df["source_file"].str.extract(r"(\d{4}_\d{2})")
df["year_month"] = pd.to_datetime(df["year_month"], format="%Y_%m")

df.head()

In [ ]:
df = df.drop_duplicates().copy()

required_cols = [
    "price",
    "squareMeters",
    "rooms",
    "floor",
    "floorCount",
    "city"
]

df = df.dropna(subset=required_cols).copy()

df = df[df["price"] > 0]
df = df[df["squareMeters"] > 0]
df = df[df["rooms"] > 0]
df = df[df["floor"] <= df["floorCount"]]

df["rooms"] = df["rooms"].astype(int)
df["floor"] = df["floor"].astype(int)
df["floorCount"] = df["floorCount"].astype(int)
df["squareMeters"] = df["squareMeters"].astype(float)
df["price"] = df["price"].astype(float)

In [ ]:
df["price_per_m2"] = df["price"] / df["squareMeters"]
df["relative_floor"] = df["floor"] / df["floorCount"]
df["m2_per_room"] = df["squareMeters"] / df["rooms"]

df["is_top_floor"] = (df["floor"] == df["floorCount"]).astype(int)

# W tych danych parter to floor == 1, nie floor == 0
df["is_ground_floor"] = (df["floor"] == 1).astype(int)

df["building_height"] = pd.cut(
    df["floorCount"],
    bins=[0, 4, 8, 50],
    labels=["niski", "średni", "wysoki"]
)

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/mieszkania_clean.csv", index=False) 